# scRNA-seq: macrophage subtypes (Fig. 2F, 2G; Supplementary Fig. S6E)
- **Fig. 2F** UMAP of macrophage subpopulations
- **Fig. 2G** macrophage subpopulation proportions per group
- **Supplementary Fig. S6E** macrophage marker dot plot

In [ ]:
import os

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns

from sc_utils import toCount_HVG_Scale

%matplotlib inline
plt.rcParams['pdf.fonttype'] = 42
sc.settings.verbosity = 3
sc.logging.print_header()
sc.set_figure_params(dpi = 100, dpi_save = 300, transparent=True, color_map = 'OrRd')

DATA_DIR = '../data'   # Melanoma_scRNAseq_data.h5ad (GEO: GSE289587 / Zenodo)

In [ ]:
adata = sc.read(f"{DATA_DIR}/Melanoma_scRNAseq_data.h5ad")
adata

## Subset and re-embed macrophages (Harmony by sample)

In [ ]:
Macro = adata[adata.obs['Level2_annotation'].str.startswith('MP')].copy()

In [ ]:
Macro = toCount_HVG_Scale(Macro)

In [ ]:
sc.tl.pca(Macro)
sc.pl.pca_variance_ratio(Macro, log=True)

In [ ]:
sc.pp.neighbors(Macro, n_pcs=20)
sc.tl.umap(Macro)

In [ ]:
sc.external.pp.harmony_integrate(Macro, key = 'sample', max_iter_harmony = 30)

In [ ]:
sc.pp.neighbors(Macro, n_pcs=50, use_rep = 'X_pca_harmony')
sc.tl.umap(Macro)

## Fig. 2F — UMAP

In [ ]:
sc.pl.umap(Macro, color = 'Level2_annotation', frameon = False, title = '')

## Fig. 2G — Proportions

In [ ]:
proportions = Macro.obs.groupby(['sample', 'Level2_annotation']).size().unstack(fill_value=0)
proportions = proportions.div(proportions.sum(axis=1), axis=0)
proportions = proportions * 100

In [ ]:
cell_type_order = 'MP_C1qc+,MP_Folr2+,MP_Il1b+,MP_Mki67+,MP_Isg15+,MP_apTAM'.split(',')
proportions = proportions[cell_type_order]

In [ ]:
proportions_with_group = proportions.reset_index()
proportions_with_group['group'] = proportions_with_group['sample'].map({
    'S1': 'Mattress', 'S2': 'Mattress', 'Sponge_1': 'Mattress', 'Sponge_3': 'Mattress',
    'T1': 'Running', 'T2': 'Running', 'Tread_1': 'Running', 'Tread_2': 'Running', 'Tread_3': 'Running'
})

In [ ]:
melted_data = pd.melt(proportions_with_group, id_vars=['sample', 'group'], var_name='cell_type', value_name='proportion')
melted_data['cell_type'] = pd.Categorical(melted_data['cell_type'], categories=cell_type_order, ordered=True)

In [ ]:
plt.figure(figsize=(20, 10))

g = sns.barplot(
    data=melted_data,
    x="cell_type",
    y="proportion",
    hue="group",
    palette=['#D4D4D4CC', '#0000FFCC'],
    width=0.8
)

g.grid(False)

g.spines['left'].set_color('black')
g.spines['bottom'].set_color('black')
g.spines['left'].set_linewidth(2)
g.spines['bottom'].set_linewidth(2)

g.spines['top'].set_visible(False)
g.spines['right'].set_visible(False)

plt.xticks(rotation=60, fontsize=20)

plt.yticks(fontsize=20)

# y축 레이블 설정
plt.ylabel("Cell type proportion (%)", fontsize=24)
plt.xlabel("", fontsize=24)

plt.legend(title="Group", bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=20)
plt.setp(g.get_legend().get_title(), fontsize=20)

for i in range(1, len(g.get_xticks())):
    g.axvline(x=i-0.5, color='gray', linestyle=(0, (5, 10)), linewidth=2)

plt.tight_layout()
plt.show()

## Supplementary Fig. S6E — Marker genes

In [ ]:
sc.pl.dotplot(Macro, ['C1qc', 'Folr2', 'Il1b', 'Isg15', 'Mki67', 'Cd74'], groupby='Level2_annotation', standard_scale='var')